<a href="https://colab.research.google.com/github/vedikakapoor27/mini_gpt/blob/main/mini_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"vedika2704","key":"e666cde1f74a2cbcc26a09dd8a9ea120"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle
!chmod 600 ~/.kaggle/kaggle.json

mkdire here is creating a hidden folder (.kaggle)
Kaggle needs this folder to store your API key

!cp
Copies your uploaded kaggle.json into that folder

 Why?
So Kaggle can authenticate (log in)

Changes file permissions

600 = only YOU can read/write it

 Why?
Kaggle requires this for security
(otherwise it throws error)

In [ ]:
!kaggle datasets download -d kingburrito666/shakespeare-plays
!unzip shakespeare-plays.zip


Dataset URL: https://www.kaggle.com/datasets/kingburrito666/shakespeare-plays
License(s): unknown
100% 4.55M/4.55M [00:00<00:00, 85.0MB/s]

Archive:  shakespeare-plays.zip
  inflating: Shakespeare_data.csv    
  inflating: alllines.txt            
  inflating: william-shakespeare-black-silhouette.jpg  


-d means dataset so it downloads the dataset and extracts all the data

In [ ]:
import os
print(os.listdir())


['.config', 'alllines.txt', 'shakespeare-plays.zip', 'william-shakespeare-black-silhouette.jpg', 'kaggle.json', 'Shakespeare_data.csv', 'sample_data']


this will show all the files in current folder


alllines.txt → FULL TEXT DATA (you will use this)
Shakespeare_data.csv → structured data (not needed for GPT here)
kaggle.json → your login key
sample_data → default Colab folder

In [ ]:
#read it in to inspect it
with open('alllines.txt','r',encoding='utf-8') as f:
  text=f.read()



In [ ]:
print("length of datasets in characters:",len(text))

length of datasets in characters: 4583798


In [ ]:
print(text[:993])

"ACT I"
"SCENE I. London. The palace."
"Enter KING HENRY, LORD JOHN OF LANCASTER, the EARL of WESTMORELAND, SIR WALTER BLUNT, and others"
"So shaken as we are, so wan with care,"
"Find we a time for frighted peace to pant,"
"And breathe short-winded accents of new broils"
"To be commenced in strands afar remote."
"No more the thirsty entrance of this soil"
"Shall daub her lips with her own children's blood,"
"Nor more shall trenching war channel her fields,"
"Nor bruise her flowerets with the armed hoofs"
"Of hostile paces: those opposed eyes,"
"Which, like the meteors of a troubled heaven,"
"All of one nature, of one substance bred,"
"Did lately meet in the intestine shock"
"And furious close of civil butchery"
"Shall now, in mutual well-beseeming ranks,"
"March all one way and be no more opposed"
"Against acquaintance, kindred and allies:"
"The edge of war, like an ill-sheathed knife,"
"No more shall cut his master. Therefore, friends,"
"As far as to the sepulchre of Christ,"



In [23]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print('' .join(chars))
print(vocab_size)

	
 !"$'(),-.0123456789:?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz
78


set(text) → gets unique characters (removes duplicates)
list(...) → converts them into a list
sorted(...) → arranges characters in order

vocab_size = len(chars)
👉 counts total number of unique characters

print(''.join(chars))
👉 shows all characters together (for checking)
print(vocab_size)
👉 shows how many characters exist

The model needs:

all possible characters
total number of characters

👉 to convert text into numbers and make predictions

In [24]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i, ch in enumerate(chars)}
encode=lambda s: [stoi[c] for c in s]# encoder: take a string, output a list of integers
decode=lambda l: ''.join([itos[i] for i in l])# decoder: take a list of integers, output a stri
print(encode("hii there"))
print(decode(encode("hii there")))

[59, 60, 60, 2, 71, 59, 56, 69, 56]
hii there


# 📌 Mini GPT – Data Preprocessing & Tokenization Summary

## 🔹 What this part does

This step prepares raw text data so that the model can understand it.
Since neural networks cannot process text directly, we convert characters into numbers.

---

## 🔹 Step-by-step explanation

### 1. Create vocabulary (unique characters)

We first extract all unique characters from the dataset.

```python
chars = sorted(list(set(text)))
vocab_size = len(chars)
```

👉 `chars` → list of unique characters
👉 `vocab_size` → total number of unique characters

---

### 2. Create mappings

```python
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
```

👉 `stoi` (string → integer)

* Converts characters to numbers
* Example: `'a' → 0`

👉 `itos` (integer → string)

* Converts numbers back to characters
* Example: `0 → 'a'`

---

### 3. Encoding (text → numbers)

```python
encode = lambda s: [stoi[c] for c in s]
```

👉 Converts a string into a list of integers
Example:

```python
encode("hi")
# Output: [7, 8] (depends on mapping)
```

---

### 4. Decoding (numbers → text)

```python
decode = lambda l: ''.join([itos[i] for i in l])
```

👉 Converts numbers back into readable text
Example:

```python
decode([7, 8])
# Output: "hi"
```

---

### 5. Test encoding & decoding

```python
print(encode("hii there"))
print(decode(encode("hii there")))
```

👉 Ensures both functions work correctly

---

## 🔹 Why this is needed

* Models cannot understand text directly ❌
* They only understand numbers ✅
* So we convert text → numbers before training

---

## 🔹 Important concept

This step is called:
👉 **Character-level Tokenization**

---

## 🔹 Where this fits in pipeline

1. Load dataset
2. Tokenization (this step) ✅
3. Convert to tensor
4. Create batches
5. Train model

---

## 🔹 Final takeaway

👉 Tokenization = preprocessing
👉 Training = learning patterns

---

## 🔹 Interview-ready line

“I implemented character-level tokenization by mapping each unique character to an integer index and used it to prepare data for training a GPT-style model.”

---


In [25]:
import torch
data=torch.tensor(encode(text) , dtype=torch.long)
print(data.shape,data.dtype)
print(data[:993])

torch.Size([4583798]) torch.int64
tensor([ 4, 24, 26, 43,  2, 32,  4,  1,  4, 42, 26, 28, 37, 28,  2, 32, 11,  2,
        35, 66, 65, 55, 66, 65, 11,  2, 43, 59, 56,  2, 67, 52, 63, 52, 54, 56,
        11,  4,  1,  4, 28, 65, 71, 56, 69,  2, 34, 32, 37, 30,  2, 31, 28, 37,
        41, 48,  9,  2, 35, 38, 41, 27,  2, 33, 38, 31, 37,  2, 38, 29,  2, 35,
        24, 37, 26, 24, 42, 43, 28, 41,  9,  2, 71, 59, 56,  2, 28, 24, 41, 35,
         2, 66, 57,  2, 46, 28, 42, 43, 36, 38, 41, 28, 35, 24, 37, 27,  9,  2,
        42, 32, 41,  2, 46, 24, 35, 43, 28, 41,  2, 25, 35, 44, 37, 43,  9,  2,
        52, 65, 55,  2, 66, 71, 59, 56, 69, 70,  4,  1,  4, 42, 66,  2, 70, 59,
        52, 62, 56, 65,  2, 52, 70,  2, 74, 56,  2, 52, 69, 56,  9,  2, 70, 66,
         2, 74, 52, 65,  2, 74, 60, 71, 59,  2, 54, 52, 69, 56,  9,  4,  1,  4,
        29, 60, 65, 55,  2, 74, 56,  2, 52,  2, 71, 60, 64, 56,  2, 57, 66, 69,
         2, 57, 69, 60, 58, 59, 71, 56, 55,  2, 67, 56, 52, 54, 56,  2, 71, 66,
      

. Import PyTorch
import torch

👉 Imports PyTorch library used for building and training models.

2. Convert text → tensor
data = torch.tensor(encode(text), dtype=torch.long)

👉 Step breakdown:

encode(text) → converts text into list of integers
torch.tensor(...) → converts list into tensor
dtype=torch.long → ensures integers (required for embeddings)

✔ Output: numerical representation of full dataset

3. Check shape & datatype
print(data.shape, data.dtype)

👉 Shows:

shape → total number of characters
dtype → integer type (int64)
4. View sample data
print(data[:1000])

👉 Displays first 1000 elements of dataset
✔ Helps verify encoding worked correctly

🔹 Key concept

👉 Text → Numbers → Tensor

Model understands only numbers, not text.

🔹 Why this is important
Neural networks work on tensors
This step prepares data for training
Required before batching and model input
🔹 One-line summary

Converts tokenized text into a PyTorch tensor so it can be used as input for training the GPT model.

In [26]:
n=int(0.9*len(data))# Let's now split up the data into train and validation sets
train_data=data[:n]
val_data=data[n:]

In [27]:
block_size=8
train_data[:block_size+1]

tensor([ 4, 24, 26, 43,  2, 32,  4,  1,  4])